# Extract word embeddings

Embed the `item` column of each `design_data_*.csv` with **BAAI/bge-large-en-v1.5** (1024-dim).

Reads the three design files in `Materials/`, embeds each item, and writes one output CSV per dataset (`embeddings_<cat>_bge.csv`) carrying the `ID`/`item`/`crit`/`training` metadata plus 1024 embedding columns.

In [1]:
from pathlib import Path
import pandas as pd
from sentence_transformers import SentenceTransformer

c:\Users\dizydorc\OneDrive\University\Project Github Repositories\Estimation processes in real-world domains\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Setup
FILES = {
    "mammals":   "design_data_mammals.csv",
    "countries": "design_data_countries.csv",
    "food":      "design_data_food.csv",
}

MODEL_NAME = "BAAI/bge-large-en-v1.5"   # 1024-dim
KEEP_COLS  = ["ID", "item", "crit", "training"]   

# Context to disambiguate bare words (e.g. "bat" the mammal vs. baseball bat).
# {item} is the item, {cat} the singular category ("mammal"/"country"/"food").
# Set to "{item}" for no context.
CONTEXT_TEMPLATE = "the {cat} {item}"

In [3]:
# First run downloads the model (~1.3 GB) to the Hugging Face cache; then offline.
model = SentenceTransformer(MODEL_NAME)

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 3804.39it/s]


In [4]:
for cat, fname in FILES.items():
    df = pd.read_csv("../../Materials/" + fname, sep=";", decimal=",", encoding="utf-8-sig")
    df.columns = [c.strip() for c in df.columns]

    items = df["item"].astype(str).str.strip()
    texts = [CONTEXT_TEMPLATE.format(item=w, cat=cat.rstrip("s")) for w in items]
    print(texts)
    vecs = model.encode(
        texts,
        normalize_embeddings = True,
        batch_size           = 32,
        show_progress_bar    = True,
    )

    emb = pd.DataFrame(vecs, columns=[f"emb_{i+1}" for i in range(vecs.shape[1])])
    out = pd.concat([df[KEEP_COLS].reset_index(drop=True), emb], axis=1)

    out_path = f"../../Data/Embeddings/embeddings_{cat}_bge.csv"
    out.to_csv(out_path, index=False) 
    print(f"{cat:>9}: {out.shape[0]} items -> {out.shape[1] - len(KEEP_COLS)} dims  "
          f"({out_path})")

['the mammal Aoudad', 'the mammal Brazilian tapir', 'the mammal American bison', 'the mammal Giant anteater', 'the mammal Ibex', "the mammal Thomson's gazelle", 'the mammal Llama', "the mammal Kirk's dik-dik", 'the mammal Red panda', 'the mammal African buffalo', 'the mammal Bactrian camel', 'the mammal Dromedary', 'the mammal Mule deer', 'the mammal Caribou and reindeer', 'the mammal Giraffe', 'the mammal Hippopotamus', 'the mammal Wart hog', 'the mammal Wild boar', 'the mammal Gray wolf', 'the mammal African wild dog', 'the mammal Raccoon dog', 'the mammal Bat-eared fox', 'the mammal Cheetah', 'the mammal Lion', 'the mammal Leopard', 'the mammal Tiger', 'the mammal Dwarf mongoose', 'the mammal Meerkat', 'the mammal Spotted hyena', 'the mammal Sea otter', 'the mammal Least weasel', 'the mammal Steller sea lion', 'the mammal Harbor seal', 'the mammal Common raccoon', 'the mammal Brown bear', 'the mammal Polar bear', 'the mammal Humpback whale', 'the mammal Bottlenosed dolphin', 'the ma

Batches:   0%|          | 0/3 [00:00<?, ?it/s]

Batches: 100%|██████████| 3/3 [00:04<00:00,  1.37s/it]


  mammals: 80 items -> 1024 dims  (../../Data/Embeddings/embeddings_mammals_bge.csv)
['the countrie China', 'the countrie India', 'the countrie United States', 'the countrie Nigeria', 'the countrie Brazil', 'the countrie Russia', 'the countrie Mexico', 'the countrie Ethiopia', 'the countrie Egypt', 'the countrie Vietnam', 'the countrie Democratic Republic of the Congo', 'the countrie Germany', 'the countrie Iran', 'the countrie France', 'the countrie South Africa', 'the countrie Tanzania', 'the countrie Colombia', 'the countrie Argentina', 'the countrie Canada', 'the countrie Ukraine', 'the countrie Angola', 'the countrie Peru', 'the countrie Yemen', 'the countrie Ivory Coast', 'the countrie Nepal', 'the countrie Venezuela', 'the countrie Australia', 'the countrie Taiwan', 'the countrie Syria', 'the countrie Burkina Faso', 'the countrie Chile', 'the countrie Netherlands', 'the countrie Ecuador', 'the countrie Benin', 'the countrie Bolivia', 'the countrie Papua New Guinea', 'the countri

Batches: 100%|██████████| 3/3 [00:01<00:00,  1.64it/s]


countries: 80 items -> 1024 dims  (../../Data/Embeddings/embeddings_countries_bge.csv)
['the food Spaghetti', 'the food Rice', 'the food Rigatoni', 'the food Bread roll', 'the food Oat flakes', 'the food Couscous', 'the food Bread', 'the food Croissant', 'the food White toast', 'the food Bulgur', 'the food Wrap', 'the food Potatoes', 'the food Cucumber', 'the food Bell pepper', 'the food Carrots', 'the food Corn', 'the food Fried egg', 'the food Broccoli', 'the food Tomatoes', 'the food Iceberg lettuce', 'the food Cauliflower', 'the food Mushrooms', 'the food Peas', 'the food Lentils', 'the food Beans', 'the food Banana', 'the food Apple', 'the food Watermelon', 'the food Kidney beans', 'the food Orange', 'the food Mandarin', 'the food Mango', 'the food Pineapple', 'the food Strawberry', 'the food Raspberry', 'the food Pear', 'the food Apricot', 'the food Grapes', 'the food Yogurt', 'the food Curd cheese', 'the food Gouda cheese', 'the food Butter', 'the food Feta', 'the food Mozzarell

Batches: 100%|██████████| 3/3 [00:01<00:00,  2.02it/s]

     food: 80 items -> 1024 dims  (../../Data/Embeddings/embeddings_food_bge.csv)
